# Deceased Flag — Full Functionality Test Suite

Tests every behavioural contract for a session where `deceasedFlag == true`:

| # | Test | Expected |
|---|------|---------|
| 1 | Login as deceased user → token carries flag | `deceasedFlag: true` in response |
| 2 | RAG /chat — AI replies in empathetic claims mode | Empathetic tone, document checklist |
| 3 | RAG /chat — AI does NOT mutate policy (address update attempt) | Refuses / redirects to claims only |
| 4 | RAG /chat — AI does NOT run premium recalculations | Refuses / redirects |
| 5 | What-If /simulate — blocked at BFF (deceased freeze) | HTTP 403 DECEASED_FLAG_FROZEN |
| 6 | Self-Service /address POST — blocked at BFF | HTTP 403 DECEASED_FLAG_FROZEN |
| 7 | Self-Service /address GET — blocked at BFF | HTTP 403 DECEASED_FLAG_FROZEN |
| 8 | RAG retrieval context — only DEATH_CLAIMS chunks returned | Section labels contain claims content |
| 9 | Normal policyholder — What-If allowed (control) | HTTP 200 |
| 10 | Normal policyholder — Self-Service allowed (control) | HTTP 200 or proxied |

**Stack**: Node.js BFF on `:5000`, Java on `:8080`. Both must be running.
Run `companion-backend-node/server.js` and `companion-backend/start.sh` first.

In [ ]:
import requests, json, sys, textwrap

BFF  = 'http://localhost:5000'
JAVA = 'http://localhost:8080'

PASS = '\033[92m✔ PASS\033[0m'
FAIL = '\033[91m✘ FAIL\033[0m'

results = []

def check(label, condition, detail=''):
    status = PASS if condition else FAIL
    tag    = 'PASS' if condition else 'FAIL'
    results.append((label, tag))
    print(f'{status}  {label}')
    if detail:
        print(textwrap.indent(str(detail)[:400], '       '))

print('Test suite loaded. Run the cells below in order.')

---
## Setup — Register test users
Creates one deceased policyholder and one live policyholder for control tests.
Safe to run multiple times — duplicate email check will return 400 (ignored).

In [ ]:
import bcrypt, requests

# ── Deceased user ─────────────────────────────────────────────────────────────
r = requests.post(f'{BFF}/api/auth/register', json={
    'first_name': 'Thabo',
    'last_name':  'Dlamini',
    'email':      'thabo.deceased@test.com',
    'username':   'thabo_deceased',
    'password':   'Test1234!',
    'role':       'ROLE_POLICYHOLDER'
})
print('Register deceased user:', r.status_code, r.json().get('message', r.text[:120]))

# ── Live policyholder (control) ───────────────────────────────────────────────
r2 = requests.post(f'{BFF}/api/auth/register', json={
    'first_name': 'Lerato',
    'last_name':  'Mokoena',
    'email':      'lerato.live@test.com',
    'username':   'lerato_live',
    'password':   'Test1234!',
    'role':       'ROLE_POLICYHOLDER'
})
print('Register live user:    ', r2.status_code, r2.json().get('message', r2.text[:120]))

### Flip `deceased_flag = true` directly in the DB
Registration always sets `deceased_flag = false` (by design — you can't self-register as deceased).  
This cell simulates the backend system setting the flag after a death is recorded.

In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname='candor', user='postgres', password='candor123',
    host='localhost', port=5432
)
conn.autocommit = True
cur = conn.cursor()
cur.execute("UPDATE users SET deceased_flag = TRUE WHERE email = 'thabo.deceased@test.com'")
cur.execute("SELECT email, deceased_flag FROM users WHERE email IN ('thabo.deceased@test.com','lerato.live@test.com')")
rows = cur.fetchall()
for row in rows:
    print(f'  {row[0]:35s}  deceased_flag={row[1]}')
cur.close(); conn.close()
print('DB flag set.')

---
## Test 1 — Login returns `deceasedFlag: true` in token payload

In [ ]:
import base64

r = requests.post(f'{BFF}/api/auth/login', json={
    'email': 'thabo.deceased@test.com',
    'password': 'Test1234!'
})
assert r.status_code == 200, f'Login failed: {r.text}'
data = r.json()

DECEASED_TOKEN = data['token']

# Decode payload (no verification needed — we just want the claims)
payload_b64 = DECEASED_TOKEN.split('.')[1]
payload_b64 += '=' * (4 - len(payload_b64) % 4)
payload = json.loads(base64.b64decode(payload_b64))

print('Token claims:', json.dumps(payload, indent=2))
print('Response user object:', json.dumps(data['user'], indent=2))

check('T1: token.deceasedFlag == true',         payload.get('deceasedFlag') == True,      payload)
check('T1: user.deceasedFlag == true in body',  data['user'].get('deceasedFlag') == True, data['user'])

---
## Test 2 — RAG /chat activates Empathetic Claims Mode
The AI must respond with empathetic tone and surface the document checklist.

In [ ]:
headers_deceased = {'Authorization': f'Bearer {DECEASED_TOKEN}'}

r = requests.post(f'{BFF}/api/rag/chat',
    headers=headers_deceased,
    json={'question': 'My father just passed away. What do I need to submit the death claim?',
          'conversationId': 'test-deceased-001'}
)
print('Status:', r.status_code)
resp = r.json()
answer = resp.get('answer', '')
print('\nAI Answer:\n', textwrap.fill(answer, 100))

check('T2: HTTP 200 for claims chat',        r.status_code == 200)
check('T2: Empathetic tone (sorry/loss/condolence)',
      any(w in answer.lower() for w in ['sorry', 'loss', 'condolence', 'deeply', 'difficult', 'grief']),
      answer[:200])
check('T2: Document checklist present (death certificate)',
      any(w in answer.lower() for w in ['death certificate', 'certified', 'bi-1663', 'dha-1663', 'id', 'bank statement']),
      answer[:200])

---
## Test 3 — Deceased session: AI refuses address update mutation

In [ ]:
r = requests.post(f'{BFF}/api/rag/chat',
    headers=headers_deceased,
    json={'question': 'Please update my address to 12 Main Street, Sandton, 2196.',
          'conversationId': 'test-deceased-001'}
)
answer = r.json().get('answer', '')
print('Status:', r.status_code)
print('AI Answer:\n', textwrap.fill(answer, 100))

# The AI should NOT confirm an address change; should redirect to claims mode
mutation_confirmed = any(w in answer.lower() for w in ['address updated', 'address has been changed', 'successfully updated'])
redirected_to_claims = any(w in answer.lower() for w in ['claim', 'claims', 'document', 'death', 'cannot', 'not able', 'unable', 'disabled', 'frozen'])

check('T3: AI does NOT confirm address mutation', not mutation_confirmed, answer[:200])
check('T3: AI redirects toward claims guidance',  redirected_to_claims,   answer[:200])

---
## Test 4 — Deceased session: AI refuses premium recalculation

In [ ]:
r = requests.post(f'{BFF}/api/rag/chat',
    headers=headers_deceased,
    json={'question': 'Can you recalculate my premium if I increase cover to R500,000?',
          'conversationId': 'test-deceased-001'}
)
answer = r.json().get('answer', '')
print('Status:', r.status_code)
print('AI Answer:\n', textwrap.fill(answer, 100))

recalc_confirmed = any(w in answer.lower() for w in ['your new premium', 'premium would be', 'recalculated to', 'updated premium'])
check('T4: AI does NOT run premium recalculation', not recalc_confirmed, answer[:200])
check('T4: Response returned (200)',               r.status_code == 200)

---
## Test 5 — What-If /simulate blocked at BFF (DECEASED_FLAG_FROZEN)

In [ ]:
r = requests.post(f'{BFF}/api/what-if/simulate',
    headers=headers_deceased,
    json={'coverAmount': 500000, 'age': 45, 'smoker': False}
)
print('Status:', r.status_code)
print('Body:  ', r.json())

check('T5: What-If returns HTTP 403',            r.status_code == 403)
check('T5: Error code is DECEASED_FLAG_FROZEN',  r.json().get('code') == 'DECEASED_FLAG_FROZEN', r.json())

---
## Test 6 — Self-Service /address POST blocked at BFF

In [ ]:
r = requests.post(f'{BFF}/api/self-service/address',
    headers=headers_deceased,
    json={'street': '12 Main St', 'suburb': 'Sandton', 'city': 'Johannesburg', 'postalCode': '2196'}
)
print('Status:', r.status_code)
print('Body:  ', r.json())

check('T6: Address POST returns HTTP 403',         r.status_code == 403)
check('T6: Error code is DECEASED_FLAG_FROZEN',    r.json().get('code') == 'DECEASED_FLAG_FROZEN', r.json())

---
## Test 7 — Self-Service /address GET blocked at BFF

In [ ]:
r = requests.get(f'{BFF}/api/self-service/address', headers=headers_deceased)
print('Status:', r.status_code)
print('Body:  ', r.json())

check('T7: Address GET returns HTTP 403',          r.status_code == 403)
check('T7: Error code is DECEASED_FLAG_FROZEN',    r.json().get('code') == 'DECEASED_FLAG_FROZEN', r.json())

---
## Test 8 — RAG /query context restricted to DEATH_CLAIMS chunks only
The raw retrieval endpoint returns the grounded context block — verify section labels
are exclusively death/claims related (no address or premium sections).

In [ ]:
# NOTE: /api/rag/query uses standard retrieval (no deceasedFlag aware path).
# Full targeted retrieval is wired through /api/rag/chat.
# This test hits /chat and inspects the conversationId echo + checks the
# answer is grounded in claims-only content.

r = requests.post(f'{BFF}/api/rag/chat',
    headers=headers_deceased,
    json={'question': 'What documents are required for a death claim?',
          'conversationId': 'test-deceased-retrieval'}
)
answer = r.json().get('answer', '')
print('Status:', r.status_code)
print('Answer:\n', textwrap.fill(answer, 100))

# Answer should mention death-claim specific content
has_claims_content = any(w in answer.lower() for w in [
    'death certificate', 'bi-1663', 'dha-1663', 'certified',
    'claimant', 'bank statement', 'beneficiary', 'claim'
])
# Answer should NOT be pulling in premium or address content
has_irrelevant_content = any(w in answer.lower() for w in [
    'your new premium', 'address updated', 'premium recalcul'
])

check('T8: Response grounded in claims content',  has_claims_content,      answer[:200])
check('T8: No address/premium mutation content',  not has_irrelevant_content, answer[:200])

---
## Test 9 — Control: live policyholder CAN use What-If

In [ ]:
# Login as live user
r = requests.post(f'{BFF}/api/auth/login', json={
    'email': 'lerato.live@test.com', 'password': 'Test1234!'
})
assert r.status_code == 200, f'Control login failed: {r.text}'
LIVE_TOKEN = r.json()['token']
headers_live = {'Authorization': f'Bearer {LIVE_TOKEN}'}

# Verify live token has deceasedFlag == false
payload_b64 = LIVE_TOKEN.split('.')[1]
payload_b64 += '=' * (4 - len(payload_b64) % 4)
live_payload = json.loads(base64.b64decode(payload_b64))
print('Live token deceasedFlag:', live_payload.get('deceasedFlag'))

# Hit What-If
r2 = requests.post(f'{BFF}/api/what-if/simulate',
    headers=headers_live,
    json={'coverAmount': 500000, 'age': 35, 'smoker': False}
)
print('What-If status:', r2.status_code)
print('What-If body:  ', str(r2.json())[:300])

check('T9: Live token.deceasedFlag == false',       live_payload.get('deceasedFlag') == False)
check('T9: What-If allowed for live user (not 403)', r2.status_code != 403, r2.json())

---
## Test 10 — Control: live policyholder CAN use Self-Service address

In [ ]:
r = requests.post(f'{BFF}/api/self-service/address',
    headers=headers_live,
    json={'street': '45 Oak Ave', 'suburb': 'Rosebank', 'city': 'Johannesburg', 'postalCode': '2196'}
)
print('Self-Service status:', r.status_code)
print('Body:               ', str(r.json())[:300])

check('T10: Address POST allowed for live user (not 403)', r.status_code != 403, r.json())

---
## Test 11 — Direct Java /api/rag/chat with manually crafted JWT
Bypasses the BFF and hits Java directly with a token whose `deceasedFlag == true`.
Confirms the Java-layer system prompt activation is independent of the BFF.

In [ ]:
import jwt as pyjwt, datetime

SECRET = 'QvXkTsAbGP7NNBHeK1RbAWaPI//n703SCJmldlUfaAw='

java_token = pyjwt.encode(
    {
        'sub':          'thabo-test-001',
        'policyId':     'POL-TEST-DECEASED',
        'role':         'ROLE_POLICYHOLDER',
        'deceasedFlag': True,
        'iss':          'https://companion.candor.local/mock-idp',
        'aud':          'candor-life-companion',
        'exp':          datetime.datetime.utcnow() + datetime.timedelta(minutes=15),
        'iat':          datetime.datetime.utcnow()
    },
    SECRET,
    algorithm='HS256'
)

r = requests.post(f'{JAVA}/api/rag/chat',
    headers={'Authorization': f'Bearer {java_token}', 'Content-Type': 'application/json'},
    json={'question': 'I need help with my late mother\'s death claim. Where do I start?',
          'conversationId': 'java-direct-deceased-001'}
)
print('Java direct status:', r.status_code)
answer = r.json().get('answer', '')
print('Answer:\n', textwrap.fill(answer, 100))

check('T11: Java direct chat returns 200',
      r.status_code == 200)
check('T11: Java activates empathetic tone',
      any(w in answer.lower() for w in ['sorry', 'loss', 'condolence', 'deeply', 'difficult']),
      answer[:200])
check('T11: Java response mentions claims requirements',
      any(w in answer.lower() for w in ['claim', 'document', 'certificate', 'certified', 'submit']),
      answer[:200])

---
## Test 12 — Multi-turn memory: deceased session retains grief context
Turn 1 establishes grief context. Turn 2 asks a follow-up — AI must NOT ask user to repeat the deceased's name.

In [ ]:
conv_id = 'test-deceased-memory-001'

# Turn 1
r1 = requests.post(f'{BFF}/api/rag/chat',
    headers=headers_deceased,
    json={'question': 'My husband John passed away last week. What do I need to claim?',
          'conversationId': conv_id}
)
a1 = r1.json().get('answer', '')
print('Turn 1:\n', textwrap.fill(a1, 100))

# Turn 2
r2 = requests.post(f'{BFF}/api/rag/chat',
    headers=headers_deceased,
    json={'question': 'What happens after I submit those documents?',
          'conversationId': conv_id}
)
a2 = r2.json().get('answer', '')
print('\nTurn 2:\n', textwrap.fill(a2, 100))

# AI should NOT ask who passed away again
asks_to_repeat = any(p in a2.lower() for p in ['who passed', 'what is the name', 'could you tell me who', 'please provide the name'])

check('T12: Turn 1 — empathetic response to grief',    r1.status_code == 200)
check('T12: Turn 2 — AI does not ask to repeat name',  not asks_to_repeat, a2[:200])
check('T12: Turn 2 — continues claims guidance',
      any(w in a2.lower() for w in ['claim', 'assess', 'process', 'submit', 'document', 'review', 'payout']),
      a2[:200])

---
## Summary

In [ ]:
print('\n' + '='*55)
print('  DECEASED FLAG TEST RESULTS')
print('='*55)
passed = sum(1 for _, s in results if s == 'PASS')
failed = sum(1 for _, s in results if s == 'FAIL')
for label, status in results:
    icon = '✔' if status == 'PASS' else '✘'
    print(f'  {icon}  {label}')
print('='*55)
print(f'  {passed} passed  |  {failed} failed  |  {len(results)} total')
print('='*55)